Go 并发面试的第一条主线，一定不是上来背 `go func(){}` 怎么写，而是要理解 Go 为什么能用很简单的写法支撑大量并发。

这一章主要解决三个问题：

第一，goroutine 是什么。
Java 里最基础的问题是线程怎么创建、线程有哪些状态、start 和 run 有什么区别；Go 里对应的问题就是 goroutine 和线程有什么区别、goroutine 为什么轻量、goroutine 会不会泄漏、goroutine 越多越好吗。

第二，GMP 调度模型是什么。
Go 的并发能力并不是因为 goroutine 本身神奇，而是因为 runtime 在 goroutine 和操作系统线程之间做了一层调度。G、M、P 分别代表 goroutine、machine、processor。理解 GMP，才能讲清楚为什么 Go 能用同步阻塞风格写高并发网络服务。

第三，channel 是什么。
Go 里有一句经典的话：不要通过共享内存来通信，而要通过通信来共享内存。channel 就是 Go 原生提供的 goroutine 通信机制。面试里常问无缓冲 channel、有缓冲 channel、close 语义、nil channel、select、channel 泄漏、channel 和 mutex 怎么选。

这一章最终要形成一句话：

Go 并发的核心不是“无限开 goroutine”，而是通过 goroutine 表达并发任务，通过 GMP 调度器把大量 goroutine 复用到少量 OS 线程上执行，通过 channel 在 goroutine 之间做同步和通信。

---

# 一、goroutine 基础

## 【文本块 2】Q：goroutine 是什么？

goroutine 是 Go runtime 管理的轻量级执行单元。

从写法上看，启动一个 goroutine 很简单：

```go
go func() {
    // 并发执行的逻辑
}()
```

只要在函数调用前加上 `go` 关键字，这个函数就会在新的 goroutine 中执行。

但是面试时不能只说“goroutine 是轻量级线程”。更准确的说法是：

> goroutine 是 Go runtime 管理的用户态轻量级并发单元，它最终仍然需要运行在操作系统线程上，但它的创建、销毁、调度都主要由 Go runtime 负责，而不是直接交给操作系统内核管理。

和 Java Thread 相比，goroutine 的特点是：

1. 创建成本更低。
   OS thread 往往需要较大的固定栈空间，而 goroutine 初始栈很小，可以按需增长。

2. 调度成本更低。
   OS thread 的调度由内核负责，涉及用户态和内核态切换；goroutine 的调度主要在 Go runtime 中完成，成本更低。

3. 数量可以更多。
   一个 Go 程序可以创建成千上万甚至更多 goroutine，但这不代表 goroutine 可以无限创建。

4. 编程模型简单。
   Go 可以用同步阻塞风格写并发代码，不需要像传统异步回调那样把逻辑拆得很碎。

面试里可以这样回答：

goroutine 是 Go runtime 调度的轻量级并发执行单元。它不是操作系统线程，但最终会被调度到 OS thread 上执行。相比直接创建线程，goroutine 初始栈更小，创建和切换成本更低，所以 Go 可以用大量 goroutine 来表达并发任务。

---

## 【代码块 1】启动一个 goroutine

```go
package main

import (
    "fmt"
    "time"
)

func main() {
    go func() {
        fmt.Println("hello from goroutine")
    }()

    time.Sleep(100 * time.Millisecond)
    fmt.Println("hello from main")
}
```

---

## 【文本块 3】代码解释

这里 `go func(){}` 会启动一个新的 goroutine。

主 goroutine 会继续往下执行，不会等待子 goroutine 自动结束。

所以我们用了：

```go
time.Sleep(100 * time.Millisecond)
```

让主 goroutine 等一会儿，避免程序太快退出，导致子 goroutine 还没来得及打印。

但是工程里不推荐用 sleep 等 goroutine，后面应该使用 WaitGroup、channel 或 context 进行同步控制。

---

## 【文本块 4】Q：主 goroutine 退出后，其他 goroutine 会怎么样？

Go 程序的入口是 main goroutine，也就是执行 main 函数的那个 goroutine。

当 main 函数返回时，整个进程会退出。
进程退出后，其他 goroutine 不管有没有执行完，都会被直接终止。

所以，主 goroutine 不会自动等待其他 goroutine。

这点和很多初学者直觉不一样。很多人以为只要启动了 goroutine，它就会自己慢慢跑完。但实际上 main 一结束，程序就结束。

---

## 【代码块 2】主 goroutine 不会自动等待子 goroutine

```go
package main

import (
    "fmt"
    "time"
)

func main() {
    go func() {
        time.Sleep(500 * time.Millisecond)
        fmt.Println("child goroutine done")
    }()

    fmt.Println("main done")
}
```

---

## 【文本块 5】代码解释

这个程序大概率只会输出：

```text
main done
```

因为 main 很快结束，整个进程退出，子 goroutine 还没睡醒就被终止了。

正确做法是使用同步机制，比如 channel 或 sync.WaitGroup。

这一点在工程里非常重要：只要你启动了 goroutine，就必须思考它的生命周期由谁管理、什么时候退出、异常怎么处理。

---

## 【代码块 3】用 channel 等待 goroutine 完成

```go
package main

import (
    "fmt"
    "time"
)

func main() {
    done := make(chan struct{})

    go func() {
        time.Sleep(100 * time.Millisecond)
        fmt.Println("child goroutine done")
        close(done)
    }()

    <-done
    fmt.Println("main done")
}
```

---

## 【文本块 6】为什么这里用 chan struct{}？

`chan struct{}` 经常用来表示信号，而不是传递具体数据。

因为 `struct{}` 是空结构体，不占额外存储空间。
当我们只关心“完成了”“可以退出了”“通知到了”，不关心具体值时，就可以用 `chan struct{}`。

这里 close(done) 表示任务完成。
主 goroutine 通过 `<-done` 阻塞等待，直到 done 被关闭或收到值。

---

# 二、goroutine 和线程的区别

## 【文本块 7】Q：goroutine 和线程有什么区别？

goroutine 和线程可以从几个维度对比。

第一，调度者不同。
线程由操作系统内核调度。
goroutine 由 Go runtime 调度，再映射到操作系统线程上执行。

第二，栈大小不同。
传统线程通常有较大的固定栈空间，比如 MB 级别。
goroutine 初始栈很小，通常只有 KB 级别，并且可以按需增长和收缩。

第三，创建和切换成本不同。
线程创建和上下文切换需要内核参与，成本较高。
goroutine 的创建和调度大多在用户态 runtime 内完成，成本更低。

第四，数量级不同。
一个进程创建几千个线程就可能很吃力。
Go 程序创建几万甚至几十万 goroutine 在某些场景下是可行的。

第五，使用方式不同。
线程模型下，开发者通常要更关心线程池、线程数、上下文切换。
goroutine 模型下，开发者更关注任务生命周期、通信、取消、并发度控制和泄漏问题。

面试里可以这样回答：

线程是操作系统调度的执行单元，goroutine 是 Go runtime 调度的轻量级执行单元。goroutine 初始栈更小，可以动态增长，创建和切换成本更低。Go runtime 会把大量 goroutine 调度到少量 OS 线程上执行，这就是 Go 能支撑高并发的重要原因。但 goroutine 也不是免费资源，创建过多仍然会带来内存、调度和 GC 压力。

---

## 【文本块 8】追问：goroutine 为什么轻量？

goroutine 轻量主要体现在两个方面。

第一，栈轻量。
OS thread 的栈一般比较大，而且很多语言里线程栈是固定大小。
goroutine 初始栈很小，并且可以动态扩容。也就是说，goroutine 不需要一开始就为最坏情况预留大量栈空间。

第二，调度轻量。
线程切换由内核调度，需要进入内核态，保存和恢复线程上下文。
goroutine 的调度由 Go runtime 在用户态完成，很多调度行为不需要内核参与。

但是，轻量不等于没有成本。

每个 goroutine 至少需要：

* 栈空间
* goroutine 描述结构
* 调度队列管理成本
* 可能的 channel、锁、timer、context 等关联资源

所以工程里不要无脑为每个小任务都无限制创建 goroutine。高并发场景还要考虑 worker pool、semaphore、限流和 backpressure。

---

## 【文本块 9】Q：goroutine 越多越好吗？

不是。

goroutine 的确比线程轻量，但它不是零成本。

goroutine 过多可能导致几个问题：

第一，内存占用上升。
每个 goroutine 都有自己的栈和元信息。几十万个 goroutine 会占用明显内存。

第二，调度压力上升。
runtime 需要管理大量可运行、阻塞、等待中的 goroutine。goroutine 数量过多时，调度开销会增加。

第三，GC 压力上升。
goroutine 栈上可能有引用，GC 需要扫描这些栈。goroutine 越多，扫描成本可能越高。

第四，下游资源被打爆。
如果每个请求都开 goroutine 访问数据库、Redis、第三方接口，而没有并发限制，很容易把连接池、下游服务或机器资源打爆。

第五，goroutine 泄漏更难排查。
goroutine 如果因为 channel 阻塞、context 没取消、select 永远等不到事件而不退出，就会泄漏。

所以 Go 并发设计的重点不是“能开多少 goroutine”，而是“如何控制 goroutine 的生命周期和并发度”。

面试里可以这样回答：

goroutine 很轻量，但不是越多越好。它仍然占用内存和调度资源，数量过多还会增加 GC 扫描成本。工程中通常要用 channel、WaitGroup、context、worker pool 或 semaphore 控制并发度，避免 goroutine 泄漏和下游资源被打爆。

---

# 三、goroutine 泄漏

## 【文本块 10】Q：什么是 goroutine 泄漏？

goroutine 泄漏是指 goroutine 启动后，因为某种原因永远阻塞或无法退出，导致它一直占用资源。

常见泄漏原因包括：

1. 读 channel 但没人写。
2. 写 channel 但没人读。
3. channel 没关闭，range 永远不结束。
4. context 没有传递或没有监听取消信号。
5. select 等待的 case 永远不会触发。
6. 定时器、ticker 没有停止。
7. 后台 goroutine 没有退出条件。
8. HTTP/RPC 调用没有超时，导致 goroutine 长期卡住。

goroutine 泄漏本质上和内存泄漏类似。
虽然泄漏的是 goroutine，但 goroutine 栈里可能引用大量对象，这些对象也无法被 GC 回收，最终导致内存上涨、调度变慢、服务异常。

---

## 【代码块 4】典型 goroutine 泄漏：读 channel 永远阻塞

```go
package main

import (
    "fmt"
    "time"
)

func main() {
    ch := make(chan int)

    go func() {
        v := <-ch
        fmt.Println("received:", v)
    }()

    // 没有人往 ch 里发送数据
    time.Sleep(100 * time.Millisecond)
    fmt.Println("main done")
}
```

---

## 【文本块 11】代码解释

子 goroutine 卡在：

```go
v := <-ch
```

因为没有任何 goroutine 往 ch 发送数据。

在这个简单程序里，main 退出后进程结束，问题不明显。
但在一个长期运行的服务中，这种 goroutine 会一直挂着，形成泄漏。

---

## 【代码块 5】用 context 避免 goroutine 永久阻塞

```go
package main

import (
    "context"
    "fmt"
    "time"
)

func worker(ctx context.Context, ch <-chan int) {
    select {
    case v := <-ch:
        fmt.Println("received:", v)
    case <-ctx.Done():
        fmt.Println("worker canceled:", ctx.Err())
        return
    }
}

func main() {
    ch := make(chan int)

    ctx, cancel := context.WithTimeout(context.Background(), 100*time.Millisecond)
    defer cancel()

    go worker(ctx, ch)

    time.Sleep(200 * time.Millisecond)
    fmt.Println("main done")
}
```

---

## 【文本块 12】工程建议：启动 goroutine 一定要问三个问题

每次你写：

```go
go func() {
    ...
}()
```

都应该问自己三个问题：

第一，它什么时候退出？
如果没有明确退出条件，就有泄漏风险。

第二，它阻塞在哪里？
如果它阻塞在 channel、锁、网络 IO、timer、select 上，就要确认这些等待是否一定能结束。

第三，它出错了怎么办？
如果 goroutine 内部 panic，没有 recover，可能导致整个进程崩溃。
如果 goroutine 内部返回 error，谁来接收和处理？

面试里可以这样回答：

goroutine 泄漏就是 goroutine 因为阻塞或没有退出条件而长期存活。常见原因是 channel 没人读写、range channel 没关闭、context 没取消、网络调用没超时。工程里启动 goroutine 时必须设计退出条件，通常通过 context、关闭 channel、WaitGroup、errgroup 来管理生命周期。

---

# 四、GMP 调度模型

## 【文本块 13】Q：什么是 GMP？

GMP 是 Go runtime 的调度模型。

G、M、P 分别代表：

G：goroutine。
表示一个待执行的 goroutine，里面保存了 goroutine 的栈、状态、程序计数器、等待信息等。

M：machine。
表示操作系统线程。真正执行代码的是 M，因为 CPU 调度的是 OS thread。

P：processor。
表示逻辑处理器，也可以理解为调度上下文。P 维护本地运行队列，持有执行 Go 代码所需的资源。

三者关系可以简单理解为：

G 是任务。
M 是干活的线程。
P 是调度资源和本地队列。

一个 M 想执行 Go 代码，必须绑定一个 P。
P 会从自己的本地队列或全局队列中取出 G，交给 M 执行。

如果用一句话概括 GMP：

> Go runtime 通过 P 管理可运行 goroutine 队列，通过 M 绑定 OS 线程执行 goroutine，从而把大量 G 调度到有限数量的 M 上运行。

---

## 【文本块 14】为什么需要 P？

早期 Go 调度模型里只有 G 和 M，没有 P。
M 直接从全局队列里取 G 执行。

这样会有几个问题：

第一，全局队列竞争严重。
所有线程都去抢同一个全局队列，锁竞争很大。

第二，调度局部性差。
刚创建的 goroutine、刚唤醒的 goroutine，可能被任意线程执行，缓存局部性不好。

第三，资源管理混乱。
一些执行 Go 代码需要的本地资源缺少统一承载者。

引入 P 后，每个 P 都有自己的本地运行队列。
M 绑定 P 后，优先从 P 的本地队列取 G 执行。
只有本地队列空了，才去全局队列或其他 P 的队列偷任务。

这样能减少全局锁竞争，提高调度效率和缓存局部性。

面试里可以这样回答：

P 的引入是为了解决全局队列竞争和调度局部性问题。P 持有本地运行队列和执行 Go 代码所需的上下文资源，M 必须绑定 P 才能执行 G。这样调度器可以优先使用本地队列，减少全局锁竞争，并通过 work stealing 做负载均衡。

---

## 【文本块 15】GMP 的基本调度流程

一个 goroutine 被创建后，大致流程是：

1. `go func()` 创建一个新的 G。
2. 新 G 优先放入当前 P 的本地运行队列。
3. M 绑定 P 后，从 P 的本地队列取 G 执行。
4. 如果本地队列为空，M 会尝试从全局队列取 G。
5. 如果全局队列也没有，M 会尝试从其他 P 的本地队列偷一半 G，这就是 work stealing。
6. 如果仍然没有可执行 G，M 可能进入休眠。
7. 当网络 IO、timer、channel、锁等事件唤醒 G 后，G 会重新进入可运行队列，等待调度。

这个过程的核心是：让每个 P 尽量有活干，避免某些 P 很忙、某些 P 空闲。

---

## 【文本块 16】Q：GOMAXPROCS 控制的是什么？

`GOMAXPROCS` 控制的是同时执行 Go 代码的 P 的数量。

换句话说，它决定了同一时刻最多有多少个 M 可以绑定 P 并并行执行 Go 代码。

如果机器有 8 个 CPU 核心，GOMAXPROCS 通常默认就是 8。
这意味着同一时刻最多有 8 个线程在并行执行 Go 代码。

注意，GOMAXPROCS 不是 goroutine 数量，也不是 OS thread 总数。

一个程序可以有很多 goroutine。
一个程序也可能创建超过 GOMAXPROCS 数量的 OS thread，比如系统调用、cgo、runtime 需要。
但同一时刻并行执行 Go 代码的数量主要受 P 的数量限制。

---

## 【代码块 6】查看和设置 GOMAXPROCS

```go
package main

import (
    "fmt"
    "runtime"
)

func main() {
    fmt.Println("NumCPU:", runtime.NumCPU())
    fmt.Println("GOMAXPROCS:", runtime.GOMAXPROCS(0))

    old := runtime.GOMAXPROCS(2)
    fmt.Println("old GOMAXPROCS:", old)
    fmt.Println("new GOMAXPROCS:", runtime.GOMAXPROCS(0))
}
```

---

## 【文本块 17】追问：GOMAXPROCS 越大越好吗？

不是。

对于 CPU 密集型任务，GOMAXPROCS 通常设置为 CPU 核心数附近比较合理。
如果设置得远大于核心数，并不会凭空增加 CPU 算力，反而可能增加线程竞争和调度开销。

对于 IO 密集型任务，goroutine 数量可以远大于 CPU 核心数，因为很多 goroutine 会阻塞等待 IO。
但是 GOMAXPROCS 控制的是并行执行 Go 代码的 P 数量，不是 IO 并发数本身。

工程里一般使用默认值即可，除非：

* 容器环境 CPU 配额识别有问题
* 需要压测调优
* 特殊 CPU 绑定或隔离场景
* 运行在资源受限环境

面试里可以这样回答：

GOMAXPROCS 控制 P 的数量，也就是同一时刻并行执行 Go 代码的能力。它不是越大越好。CPU 密集型任务一般接近 CPU 核心数即可；IO 并发更多依赖 goroutine 阻塞和 runtime netpoller，不应该简单通过无限增大 GOMAXPROCS 解决。

---

# 五、M 阻塞时发生什么？

## 【文本块 18】Q：如果 goroutine 发生阻塞，GMP 怎么处理？

要分情况。

第一种，goroutine 阻塞在 channel、mutex、timer、select 等 Go runtime 能感知的阻塞点。
这时 runtime 会把当前 G 挂起，M 继续绑定 P 去执行其他 G。这个过程不需要阻塞 OS thread。

第二种，goroutine 阻塞在网络 IO。
Go runtime 会使用 netpoller。网络 fd 通常是非阻塞的。如果数据没准备好，G 会被挂起，M 和 P 可以去执行其他 G。等网络事件就绪后，G 再被放回运行队列。

第三种，goroutine 阻塞在系统调用。
比如某些文件 IO、阻塞 syscall。此时 M 可能会进入系统调用阻塞。为了不浪费 P，runtime 会把 P 从这个阻塞的 M 上解绑，交给其他 M 继续执行别的 G。当系统调用返回后，原来的 G 再尝试获取 P 继续运行。

第四种，goroutine 执行 CPU 密集型死循环。
如果没有函数调用、没有安全点，早期 Go 版本可能导致长时间不被抢占。现代 Go 已经支持更强的异步抢占，可以减少这种问题，但 CPU 密集型代码仍然要注意不要长期霸占 CPU。

---

## 【文本块 19】Q：网络 IO 阻塞为什么不会阻塞线程？

Go 网络库通常会把网络 fd 设置成非阻塞模式。

当 goroutine 调用：

```go
conn.Read(buf)
```

如果数据已经准备好，就直接读取。
如果数据没准备好，runtime 不会让 OS thread 一直卡在那里等，而是把这个 goroutine 挂到 netpoller 上等待事件，然后让 M 去执行其他 goroutine。

等 epoll/kqueue/IOCP 通知这个 fd 可读，runtime 再把对应 goroutine 放回可运行队列。

所以从代码层面看，`conn.Read` 是阻塞式写法；
从 runtime 层面看，它是非阻塞 fd + 事件通知 + goroutine 挂起恢复。

这就是 Go 网络编程简单而高并发的关键。

面试里可以这样回答：

Go 网络 IO 对开发者呈现的是同步阻塞模型，但 runtime 底层使用 netpoller 和非阻塞 fd。当网络数据未就绪时，阻塞的是 goroutine，而不是长期占用 OS thread。runtime 会把线程拿去执行其他 goroutine，等网络事件就绪后再恢复原 goroutine。

---

# 六、work stealing 和 handoff

## 【文本块 20】Q：什么是 work stealing？

work stealing 是 Go 调度器的负载均衡机制。

每个 P 都有自己的本地运行队列。
如果某个 P 的本地队列空了，它绑定的 M 没有 G 可执行，就会尝试从其他 P 的本地队列偷一部分 G 过来执行。

为什么要偷一半？

因为如果只偷一个，可能很快又空了，偷取频率太高。
如果全部偷走，又会让被偷的 P 没活干。
偷一部分可以在负载均衡和局部性之间折中。

面试里可以这样回答：

work stealing 是指当某个 P 没有可运行 G 时，会从其他 P 的本地队列偷一部分 G 来执行。这样可以避免某些 P 很忙、某些 P 空闲，提高整体 CPU 利用率。

---

## 【文本块 21】Q：什么是 handoff？

handoff 通常指当 M 因系统调用等原因阻塞时，runtime 会把它绑定的 P 移交给其他 M。

因为 M 阻塞在 syscall 里时，不能继续执行 Go 代码。
如果 P 也跟着它一起被卡住，就会浪费调度资源。

所以 runtime 会让阻塞的 M 暂时失去 P，其他空闲 M 接管这个 P，继续执行 P 队列里的 G。

当原来的系统调用返回后，M 再尝试获取一个 P 来继续运行。如果没有空闲 P，它可能把 G 放回队列，自己休眠。

这个机制保证了：一个线程阻塞在系统调用里，不会拖死它原来绑定的 P。

---

# 七、Go 调度器抢占

## 【文本块 22】Q：Go 调度器是协作式还是抢占式？

Go 调度器经历过演进。

早期 Go 更多依赖协作式抢占。
也就是说，goroutine 只有在函数调用、channel 操作、系统调用、GC 安全点等位置才容易让出执行权。

如果一个 goroutine 长时间执行没有函数调用的 CPU 死循环，可能影响调度。

后来 Go 引入了更强的异步抢占能力。runtime 可以在更多位置打断长期运行的 goroutine，避免它一直霸占 P。

所以现代 Go 可以说已经支持抢占式调度，但理解上仍然要知道：Go 的抢占不是完全等同于操作系统线程抢占，它是 runtime 层面的调度机制，和安全点、栈扫描、信号等机制有关。

面试里可以这样回答：

Go 调度器早期更偏协作式，后来引入异步抢占，现代 Go 已经能更好地抢占长时间运行的 goroutine。这样可以避免 CPU 密集型 goroutine 长期霸占 P，提高调度公平性和 GC 可控性。

---

## 【文本块 23】GMP 面试速记版

G 是 goroutine，代表任务。
M 是 OS thread，代表真正执行代码的线程。
P 是 processor，代表调度上下文和本地运行队列。
M 必须绑定 P 才能执行 Go 代码。
GOMAXPROCS 控制 P 的数量。
P 有本地运行队列，可以减少全局队列锁竞争。
本地队列空了会从全局队列拿，或者从其他 P 偷，这叫 work stealing。
M 阻塞在 syscall 时，P 会解绑并交给其他 M，这叫 handoff。
网络 IO 阻塞时，G 会挂到 netpoller，M 不会长期阻塞。
现代 Go 支持异步抢占，避免 goroutine 长时间霸占 CPU。

一句话模板：

Go 的 GMP 模型本质上是把大量 goroutine 通过 runtime 调度到少量 OS 线程上执行。G 表示任务，M 表示线程，P 表示调度资源和本地队列。通过本地队列、work stealing、netpoller、syscall handoff 和抢占式调度，Go 在保持同步编程模型的同时获得较好的并发性能。

---

# 八、channel 基础

## 【文本块 24】Q：channel 是什么？

channel 是 Go 提供的 goroutine 之间通信和同步的机制。

可以把 channel 理解成一个类型安全的管道。

一个 goroutine 可以往 channel 发送数据：

```go
ch <- value
```

另一个 goroutine 可以从 channel 接收数据：

```go
v := <-ch
```

channel 有两类：

1. 无缓冲 channel
2. 有缓冲 channel

无缓冲 channel 要求发送方和接收方同时准备好。
有缓冲 channel 内部有队列，发送方可以在缓冲区未满时直接发送，接收方可以在缓冲区非空时直接接收。

面试里可以这样回答：

channel 是 Go 原生的 goroutine 通信机制，既能传递数据，也能做同步。无缓冲 channel 强调发送和接收同步；有缓冲 channel 在容量范围内可以解耦发送方和接收方。

---

## 【代码块 7】channel 基本使用

```go
package main

import "fmt"

func main() {
    ch := make(chan int)

    go func() {
        ch <- 42
    }()

    v := <-ch
    fmt.Println(v)
}
```

---

## 【文本块 25】代码解释

这里 ch 是无缓冲 channel。

子 goroutine 执行：

```go
ch <- 42
```

会阻塞，直到主 goroutine 执行：

```go
v := <-ch
```

主 goroutine 接收后，发送操作和接收操作同时完成。

无缓冲 channel 的核心语义是同步交接，而不是单纯存数据。

---

# 九、无缓冲 channel

## 【文本块 26】Q：无缓冲 channel 有什么特点？

无缓冲 channel 没有存储空间。

发送必须等待接收。
接收也必须等待发送。

所以无缓冲 channel 常用于：

1. goroutine 间同步。
2. 任务交接。
3. 保证某个事件已经发生。
4. 控制执行顺序。

例如：

```go
done := make(chan struct{})
```

主 goroutine 等待 `<-done`，子 goroutine 完成后 `close(done)`。

这本质上就是一个同步信号。

---

## 【代码块 8】无缓冲 channel 控制执行顺序

```go
package main

import "fmt"

func main() {
    ch := make(chan string)

    go func() {
        ch <- "step 1 done"
    }()

    msg := <-ch
    fmt.Println(msg)
    fmt.Println("step 2")
}
```

---

## 【文本块 27】追问：无缓冲 channel 为什么能做同步？

因为无缓冲 channel 的发送和接收必须同时配对。

发送方只有在接收方准备好时才能继续。
接收方只有在发送方准备好时才能继续。

所以一次无缓冲 channel 通信完成后，可以确定发送前的操作对接收方可见。

这也是 Go 内存模型中的重要同步关系之一：channel send happens-before 对应 receive 完成。

面试里可以简单说：

无缓冲 channel 不只是传值，它还建立了同步关系。发送和接收配对完成后，发送方在发送前的写入，对接收方在接收后的操作可见。

---

# 十、有缓冲 channel

## 【文本块 28】Q：有缓冲 channel 有什么特点？

有缓冲 channel 内部有固定容量的队列。

创建方式：

```go
ch := make(chan int, 3)
```

容量是 3。

发送时：

* 如果缓冲区没满，发送方可以直接写入，不阻塞。
* 如果缓冲区满了，发送方阻塞。

接收时：

* 如果缓冲区非空，接收方可以直接取出，不阻塞。
* 如果缓冲区为空，接收方阻塞。

有缓冲 channel 常用于：

1. 生产者消费者队列。
2. 控制并发度。
3. 削峰缓冲。
4. 异步结果传递。
5. worker pool 任务队列。

---

## 【代码块 9】有缓冲 channel 基本使用

```go
package main

import "fmt"

func main() {
    ch := make(chan int, 2)

    ch <- 1
    ch <- 2

    fmt.Println(<-ch)
    fmt.Println(<-ch)
}
```

---

## 【文本块 29】代码解释

channel 容量是 2。

前两次发送不会阻塞，因为缓冲区有空间。

如果再发送第三次，而没有 goroutine 接收，就会阻塞，最终可能导致 deadlock。

---

## 【代码块 10】有缓冲 channel 满了会阻塞

```go
package main

import "fmt"

func main() {
    ch := make(chan int, 2)

    ch <- 1
    ch <- 2

    // 如果取消下面这行注释，会发生死锁：
    // ch <- 3

    fmt.Println(<-ch)
    fmt.Println(<-ch)
}
```

---

## 【文本块 30】追问：有缓冲 channel 容量越大越好吗？

不是。

缓冲区太小，生产者和消费者容易互相阻塞。
缓冲区太大，可能掩盖消费能力不足的问题，导致内存堆积和延迟变高。

有缓冲 channel 的容量本质上是一种背压设计。

如果缓冲区满了，生产者被阻塞，说明下游消费能力跟不上。这个阻塞反而是一种保护，避免无限堆积。

所以工程里 channel buffer 不是越大越好，而要根据任务耗时、吞吐、内存预算和延迟要求设计。

面试里可以这样回答：

有缓冲 channel 可以解耦生产者和消费者，但容量不是越大越好。容量太大可能掩盖下游慢的问题，造成任务堆积和内存上涨；容量太小又可能导致频繁阻塞。它本质上是并发系统中的背压手段。

---

# 十一、channel 的 close 语义

## 【文本块 31】Q：close channel 是什么意思？

close channel 表示不会再有新的数据发送到这个 channel。

关闭 channel 不是清空 channel。
关闭后，已经在缓冲区里的数据仍然可以被接收。

当缓冲区数据被读完后，再继续接收，会立即返回该类型的零值，并且 ok 为 false。

常见写法：

```go
v, ok := <-ch
```

ok 为 true 表示正常接收到数据。
ok 为 false 表示 channel 已关闭且数据已读完。

---

## 【代码块 11】关闭 channel 后继续接收

```go
package main

import "fmt"

func main() {
    ch := make(chan int, 2)

    ch <- 1
    ch <- 2
    close(ch)

    v1, ok1 := <-ch
    v2, ok2 := <-ch
    v3, ok3 := <-ch

    fmt.Println(v1, ok1)
    fmt.Println(v2, ok2)
    fmt.Println(v3, ok3)
}
```

---

## 【文本块 32】代码解释

输出类似：

```text
1 true
2 true
0 false
```

前两次读到的是缓冲区里的数据。
第三次缓冲区已经空了，channel 也关闭了，所以返回 int 的零值 0，ok 为 false。

---

## 【文本块 33】Q：向关闭的 channel 发送数据会怎样？

会 panic。

```go
ch := make(chan int)
close(ch)
ch <- 1
```

这是非法操作。

所以一般遵循一个原则：

> channel 应该由发送方关闭，而不是接收方关闭。

因为发送方最清楚什么时候不会再发送数据。
接收方贸然关闭 channel，可能导致其他发送方继续发送，从而 panic。

---

## 【代码块 12】不要向已关闭 channel 发送数据

```go
package main

func main() {
    ch := make(chan int, 1)

    close(ch)

    // 如果取消注释，会 panic：send on closed channel
    // ch <- 1
}
```

---

## 【文本块 34】追问：重复 close channel 会怎样？

重复 close 会 panic。

```go
close(ch)
close(ch)
```

所以如果多个 goroutine 都可能关闭 channel，必须通过额外机制保证只关闭一次，比如：

* sync.Once
* 只有唯一发送方负责关闭
* context 取消替代 close
* 明确所有权设计

工程里最推荐的是设计好所有权：谁生产，谁关闭。不要让多个地方都能 close 同一个 channel。

---

# 十二、range channel

## 【文本块 35】Q：for range channel 是怎么工作的？

可以用 for range 持续从 channel 接收数据：

```go
for v := range ch {
    fmt.Println(v)
}
```

这个循环会一直读取 channel，直到 channel 被关闭并且缓冲区数据被读完。

如果 channel 永远不关闭，range 会一直阻塞。

所以使用 range channel 时，一定要确保发送方最终会 close channel。

---

## 【代码块 13】range channel

```go
package main

import "fmt"

func main() {
    ch := make(chan int)

    go func() {
        for i := 0; i < 3; i++ {
            ch <- i
        }
        close(ch)
    }()

    for v := range ch {
        fmt.Println(v)
    }

    fmt.Println("done")
}
```

---

## 【文本块 36】代码解释

子 goroutine 发送 0、1、2 后关闭 channel。

主 goroutine 用 range 接收，直到 channel 关闭并读完所有数据，循环结束。

如果去掉 `close(ch)`，主 goroutine 会一直等下一条数据，最终 deadlock。

---

# 十三、nil channel

## 【文本块 37】Q：nil channel 读写会怎样？

nil channel 是没有初始化的 channel。

```go
var ch chan int
```

此时 ch 是 nil。

对 nil channel：

* 发送会永久阻塞
* 接收会永久阻塞
* close 会 panic

这点非常重要。

nil channel 常见于 bug，也可以在 select 中被有意使用：把某个 channel 置为 nil，可以禁用对应 case。

---

## 【代码块 14】nil channel 示例

```go
package main

func main() {
    var ch chan int

    // 下面两行任意取消注释都会永久阻塞：
    // ch <- 1
    // <-ch

    // 下面这行会 panic：
    // close(ch)
}
```

---

## 【文本块 38】nil channel 在 select 中的用法

在 select 中，nil channel 对应的 case 永远不会被选中。

这可以用来动态启用或禁用某个 case。

例如，当某个输入 channel 已关闭后，可以把它置为 nil，避免 select 一直选中它。

这是进阶技巧，但面试中能讲出来会加分。

---

# 十四、select

## 【文本块 39】Q：select 是什么？

select 用于同时等待多个 channel 操作。

语法类似 switch，但每个 case 是 channel 的发送或接收。

如果多个 case 同时就绪，Go 会伪随机选择一个执行。
如果没有 case 就绪，并且没有 default，select 会阻塞。
如果有 default，select 不会阻塞，会立即执行 default。

select 常见用途：

1. 同时监听多个 channel。
2. 实现超时控制。
3. 实现取消机制。
4. 非阻塞发送或接收。
5. 合并多个 channel。
6. 控制 goroutine 生命周期。

---

## 【代码块 15】select 基本使用

```go
package main

import (
    "fmt"
    "time"
)

func main() {
    ch1 := make(chan string)
    ch2 := make(chan string)

    go func() {
        time.Sleep(100 * time.Millisecond)
        ch1 <- "from ch1"
    }()

    go func() {
        time.Sleep(200 * time.Millisecond)
        ch2 <- "from ch2"
    }()

    select {
    case msg := <-ch1:
        fmt.Println(msg)
    case msg := <-ch2:
        fmt.Println(msg)
    }
}
```

---

## 【文本块 40】select 的随机性

如果多个 case 同时就绪，select 不保证按代码顺序选择，而是伪随机选择一个。

这样做是为了避免某些 case 长期饥饿。

所以不要写依赖 select case 顺序的逻辑。

---

## 【代码块 16】select + timeout

```go
package main

import (
    "fmt"
    "time"
)

func main() {
    ch := make(chan string)

    select {
    case msg := <-ch:
        fmt.Println("received:", msg)
    case <-time.After(100 * time.Millisecond):
        fmt.Println("timeout")
    }
}
```

---

## 【文本块 41】time.After 的注意事项

`time.After(d)` 会返回一个 channel，时间到了以后发送当前时间。

简单超时场景可以用 time.After。

但在高频循环中反复调用 time.After，可能产生大量 timer，带来额外开销。

更严谨的方式是使用 time.NewTimer，并在不需要时 Stop。

在现代 Go 版本里 timer 已有优化，但工程上仍要避免在超高频路径里无脑创建大量 timer。

---

## 【代码块 17】select + default 实现非阻塞接收

```go
package main

import "fmt"

func main() {
    ch := make(chan int)

    select {
    case v := <-ch:
        fmt.Println("received:", v)
    default:
        fmt.Println("no data")
    }
}
```

---

## 【文本块 42】default 的作用

如果 select 有 default，并且其他 case 都没就绪，就会立即执行 default。

这可以实现非阻塞操作。

但也要小心：如果在 for 循环里加 default，可能导致忙等，CPU 飙高。

例如：

```go
for {
    select {
    case v := <-ch:
        handle(v)
    default:
    }
}
```

这个循环会疯狂空转，浪费 CPU。

如果确实需要轮询，应该加 sleep、ticker，或者重新设计阻塞模型。

---

# 十五、channel 底层结构

## 【文本块 43】Q：channel 底层是怎么实现的？

Go channel 底层是 runtime.hchan 结构。

虽然不需要背源码字段，但面试里可以讲几个核心组成：

1. qcount：当前缓冲区里有多少元素。
2. dataqsiz：缓冲区容量。
3. buf：环形缓冲区。
4. elemsize：元素大小。
5. closed：是否关闭。
6. sendx：发送位置索引。
7. recvx：接收位置索引。
8. recvq：等待接收的 goroutine 队列。
9. sendq：等待发送的 goroutine 队列。
10. lock：保护 channel 内部状态的锁。

对于有缓冲 channel，数据会进入环形队列。

对于无缓冲 channel，如果发送方和接收方同时存在，数据可以直接从发送方拷贝给接收方，不需要经过缓冲区。

如果发送方没有接收方配对，发送 goroutine 会进入 sendq 阻塞。
如果接收方没有发送方配对，接收 goroutine 会进入 recvq 阻塞。

面试里可以这样回答：

channel 底层是 hchan，内部有环形缓冲区、发送等待队列、接收等待队列和互斥锁。发送和接收时，runtime 会根据缓冲区状态和等待队列决定是直接交接、写入缓冲区，还是把当前 goroutine 挂起到 sendq/recvq。

---

## 【文本块 44】无缓冲 channel 的发送流程

无缓冲 channel 没有缓冲区。

发送时大致分几种情况：

1. 如果 recvq 中已经有等待接收的 goroutine，直接把数据交给接收方，唤醒接收方。
2. 如果没有接收方，当前发送 goroutine 进入 sendq，阻塞等待。

接收时也是类似：

1. 如果 sendq 中已经有等待发送的 goroutine，直接从发送方拿数据，唤醒发送方。
2. 如果没有发送方，当前接收 goroutine 进入 recvq，阻塞等待。

所以无缓冲 channel 的本质是同步交接。

---

## 【文本块 45】有缓冲 channel 的发送流程

有缓冲 channel 有环形队列。

发送时：

1. 如果 recvq 中有等待接收的 goroutine，可能直接把数据交给接收方。
2. 否则如果缓冲区未满，把数据放入 buf，对 sendx 取模移动。
3. 如果缓冲区已满，当前发送 goroutine 进入 sendq 阻塞。

接收时：

1. 如果缓冲区有数据，从 buf 取出，对 recvx 取模移动。
2. 如果 sendq 中有等待发送的 goroutine，可能把发送方的数据补进缓冲区。
3. 如果缓冲区为空且没有发送方，当前接收 goroutine 进入 recvq 阻塞。

channel 底层用锁保护这些状态，所以 channel 本身是并发安全的。

---

# 十六、channel 和共享内存

## 【文本块 46】Q：channel 和 mutex 怎么选？

Go 里常说：

> Do not communicate by sharing memory; instead, share memory by communicating.

意思是：不要通过共享内存来通信，而要通过通信来共享内存。

但这不是说 mutex 不应该用。

channel 和 mutex 适合的场景不同。

channel 更适合：

1. 数据在 goroutine 之间流动。
2. 任务分发。
3. 生产者消费者。
4. 事件通知。
5. 生命周期取消。
6. pipeline。
7. 需要表达“交接”和“顺序”的场景。

mutex 更适合：

1. 保护共享状态。
2. 临界区很短。
3. 多个 goroutine 读写同一份 map、计数器、缓存。
4. 需要直接修改内存结构。
5. 状态天然属于同一个对象。

错误倾向有两种：

一种是所有并发都用 channel，导致代码绕来绕去。
另一种是所有并发都用锁，导致通信关系不清晰。

面试里可以这样回答：

如果核心是数据流动和任务交接，我会优先考虑 channel；如果核心是保护一段共享状态，我会优先考虑 mutex。channel 不是锁的替代品，mutex 也不是 channel 的替代品，关键看问题本质是通信还是共享状态保护。

---

## 【代码块 18】用 channel 做任务分发

```go
package main

import "fmt"

func worker(id int, jobs <-chan int, results chan<- int) {
    for job := range jobs {
        fmt.Println("worker", id, "processing job", job)
        results <- job * 2
    }
}

func main() {
    jobs := make(chan int, 3)
    results := make(chan int, 3)

    for w := 1; w <= 2; w++ {
        go worker(w, jobs, results)
    }

    for j := 1; j <= 3; j++ {
        jobs <- j
    }
    close(jobs)

    for a := 1; a <= 3; a++ {
        fmt.Println("result:", <-results)
    }
}
```

---

## 【文本块 47】代码解释

这里 jobs channel 用于分发任务。
多个 worker goroutine 从 jobs 读取任务。
处理完成后，把结果写入 results。

这就是一个最简化的 worker pool 雏形。

注意 worker 函数参数里：

```go
jobs <-chan int
results chan<- int
```

这是单向 channel。

`<-chan int` 表示只读 channel。
`chan<- int` 表示只写 channel。

这样可以在编译期限制函数对 channel 的使用方向，提高代码安全性。

---

# 十七、单向 channel

## 【文本块 48】Q：什么是单向 channel？

普通 channel 可以读也可以写：

```go
ch chan int
```

只读 channel：

```go
ch <-chan int
```

只写 channel：

```go
ch chan<- int
```

单向 channel 通常用于函数参数，表示这个函数只能从 channel 接收，或者只能向 channel 发送。

这不是创建了新的底层 channel，而是在类型层面对使用方向做限制。

例如：

```go
func producer(out chan<- int)
func consumer(in <-chan int)
```

这样 producer 不能从 out 里读，consumer 不能往 in 里写。

---

## 【代码块 19】单向 channel 示例

```go
package main

import "fmt"

func producer(out chan<- int) {
    for i := 0; i < 3; i++ {
        out <- i
    }
    close(out)
}

func consumer(in <-chan int) {
    for v := range in {
        fmt.Println(v)
    }
}

func main() {
    ch := make(chan int)

    go producer(ch)
    consumer(ch)
}
```

---

## 【文本块 49】单向 channel 的工程价值

单向 channel 的价值是让函数签名表达意图。

当你看到：

```go
func producer(out chan<- Task)
```

就知道这个函数只负责生产任务，不应该消费任务。

当你看到：

```go
func consumer(in <-chan Task)
```

就知道这个函数只负责消费任务，不应该关闭或写入任务 channel。

这是一种很 Go 的工程风格：用类型系统表达约束，让错误在编译期暴露。

---

# 十八、channel 常见死锁

## 【文本块 50】Q：channel 常见死锁有哪些？

第一，无缓冲 channel 同一个 goroutine 先发送再接收。

```go
ch := make(chan int)
ch <- 1
fmt.Println(<-ch)
```

发送时没有其他 goroutine 接收，会死锁。

第二，有缓冲 channel 写满后继续写，没有接收者。

```go
ch := make(chan int, 1)
ch <- 1
ch <- 2
```

第二次发送阻塞。

第三，range channel 但发送方没有 close。

```go
for v := range ch {
}
```

如果 ch 不关闭，range 永远等下去。

第四，select 等待的所有 channel 都不会就绪，并且没有 default。

第五，nil channel 读写永久阻塞。

第六，多个 goroutine 互相等待对方发送或关闭 channel。

面试里可以这样回答：

channel 死锁的本质是所有 goroutine 都在等待一个永远不会发生的 channel 操作。常见原因包括无缓冲 channel 没有配对接收、缓冲区满了没人消费、range 等待未关闭 channel、nil channel 读写、select 没有就绪 case。排查时要看每个 goroutine 阻塞在哪个 channel 操作上。

---

## 【代码块 20】不要运行：无缓冲 channel 死锁示例

```go
package main

func main() {
    ch := make(chan int)

    // 这行会阻塞，因为没有其他 goroutine 接收
    // ch <- 1

    // fmt.Println(<-ch)
}
```

---

# 十九、channel 实现生产者消费者

## 【文本块 51】Q：如何用 channel 实现生产者消费者？

生产者消费者模型很适合用 channel。

生产者负责把任务写入 channel。
消费者负责从 channel 读取任务并处理。
当所有任务生产完后，生产者关闭 channel。
消费者通过 range channel 自动退出。

关键点：

1. channel 作为任务队列。
2. 可以启动多个消费者并发处理。
3. 生产者负责 close channel。
4. 消费者不要 close 任务 channel。
5. 主 goroutine 要等待所有消费者退出。

---

## 【代码块 21】生产者消费者示例

```go
package main

import (
    "fmt"
    "sync"
)

func producer(jobs chan<- int) {
    for i := 1; i <= 5; i++ {
        jobs <- i
    }
    close(jobs)
}

func consumer(id int, jobs <-chan int, wg *sync.WaitGroup) {
    defer wg.Done()

    for job := range jobs {
        fmt.Printf("consumer %d got job %d\n", id, job)
    }
}

func main() {
    jobs := make(chan int, 2)

    var wg sync.WaitGroup

    for i := 1; i <= 3; i++ {
        wg.Add(1)
        go consumer(i, jobs, &wg)
    }

    producer(jobs)

    wg.Wait()
    fmt.Println("all consumers done")
}
```

---

## 【文本块 52】代码解释

producer 把 1 到 5 写入 jobs，然后 close(jobs)。

多个 consumer 用 range jobs 读取任务。
当 jobs 被关闭且数据读完后，range 自动结束。
每个 consumer 退出前调用 wg.Done。
主 goroutine 用 wg.Wait 等待所有 consumer 结束。

这就是 Go 并发中最常见的模式之一。

---

# 二十、channel 实现并发限制

## 【文本块 53】Q：如何用 channel 控制并发数？

可以用有缓冲 channel 当作 semaphore。

例如：

```go
sem := make(chan struct{}, 3)
```

容量是 3，表示最多允许 3 个任务同时执行。

每个任务开始前：

```go
sem <- struct{}{}
```

任务结束后：

```go
<-sem
```

如果已经有 3 个任务在执行，第 4 个任务会阻塞在发送处，直到有人释放。

---

## 【代码块 22】用 channel 控制并发数

```go
package main

import (
    "fmt"
    "sync"
    "time"
)

func main() {
    sem := make(chan struct{}, 3)

    var wg sync.WaitGroup

    for i := 1; i <= 10; i++ {
        i := i

        wg.Add(1)
        go func() {
            defer wg.Done()

            sem <- struct{}{}
            defer func() { <-sem }()

            fmt.Println("start task", i)
            time.Sleep(200 * time.Millisecond)
            fmt.Println("finish task", i)
        }()
    }

    wg.Wait()
}
```

---

## 【文本块 54】工程解释

这里最多只有 3 个 goroutine 能进入真正的任务执行区。

这是一种非常常见的并发控制方式。

适合：

* 批量请求第三方 API
* 批量处理文件
* 批量发送消息
* 限制数据库并发写入
* 控制 CPU 或 IO 压力

但是在更复杂场景中，更推荐使用成熟的 worker pool 或 errgroup + semaphore，因为还要处理错误、取消、panic、结果聚合等问题。

---

# 二十一、channel 和内存可见性

## 【文本块 55】Q：channel 能保证内存可见性吗？

能。

Go 内存模型规定，channel 操作会建立 happens-before 关系。

简单理解：

1. 对无缓冲 channel，发送操作 happens-before 对应接收操作完成。
2. 对关闭 channel，close happens-before 接收方观察到 channel 已关闭。
3. 对有缓冲 channel，也有相应的发送、接收顺序保证。

例如：

```go
x = 1
ch <- struct{}{}
```

另一个 goroutine：

```go
<-ch
fmt.Println(x)
```

只要接收到了 ch 的信号，就能看到发送前对 x 的写入。

所以 channel 不只是通信工具，也是一种同步工具。

---

## 【代码块 23】channel 保证同步可见性

```go
package main

import "fmt"

func main() {
    ch := make(chan struct{})
    x := 0

    go func() {
        x = 42
        close(ch)
    }()

    <-ch
    fmt.Println(x)
}
```

---

## 【文本块 56】代码解释

子 goroutine 先写 x，再 close(ch)。

主 goroutine `<-ch` 等待 channel 关闭后再读取 x。

由于 close channel 和接收关闭信号之间存在同步关系，主 goroutine 可以看到 x = 42。

当然，这种写法适合一次性通知。如果有复杂共享状态，还是要根据场景选择 mutex、atomic 或 channel。

---

# 二十二、channel 常见工程陷阱

## 【文本块 57】陷阱 1：多个发送方关闭同一个 channel

如果多个 goroutine 都可能 close 同一个 channel，很容易发生重复 close panic。

更推荐的设计是：

* 多个发送方只发送，不关闭。
* 由协调者等待所有发送方结束后统一关闭。
* 或者使用 context 表示取消，不用 close 数据 channel。

---

## 【代码块 24】多个发送方时由协调者关闭 channel

```go
package main

import (
    "fmt"
    "sync"
)

func main() {
    ch := make(chan int)

    var producers sync.WaitGroup

    for p := 1; p <= 3; p++ {
        p := p
        producers.Add(1)

        go func() {
            defer producers.Done()
            for i := 0; i < 2; i++ {
                ch <- p*10 + i
            }
        }()
    }

    go func() {
        producers.Wait()
        close(ch)
    }()

    for v := range ch {
        fmt.Println(v)
    }
}
```

---

## 【文本块 58】陷阱 2：向无接收者的 channel 发送导致 goroutine 堵死

很多异步代码会这样写：

```go
resultCh <- result
```

如果调用方已经超时退出，不再接收 resultCh，发送方就会永远阻塞。

解决方式：

1. resultCh 使用合适缓冲。
2. 发送时 select 监听 ctx.Done。
3. 调用方确保 drain channel。
4. 更复杂场景用 errgroup/context 管理。

---

## 【代码块 25】发送结果时监听 context

```go
package main

import (
    "context"
    "fmt"
    "time"
)

func worker(ctx context.Context, resultCh chan<- string) {
    result := "done"

    select {
    case resultCh <- result:
        fmt.Println("sent result")
    case <-ctx.Done():
        fmt.Println("worker canceled before sending result")
        return
    }
}

func main() {
    ctx, cancel := context.WithTimeout(context.Background(), 50*time.Millisecond)
    defer cancel()

    resultCh := make(chan string)

    go func() {
        time.Sleep(100 * time.Millisecond)
        worker(ctx, resultCh)
    }()

    select {
    case r := <-resultCh:
        fmt.Println("result:", r)
    case <-ctx.Done():
        fmt.Println("main timeout")
    }

    time.Sleep(200 * time.Millisecond)
}
```

---

## 【文本块 59】陷阱 3：for select default 忙等

错误写法：

```go
for {
    select {
    case v := <-ch:
        handle(v)
    default:
    }
}
```

如果 ch 没数据，这个循环会不断执行 default，导致 CPU 空转。

解决方案：

1. 去掉 default，让 goroutine 阻塞等待。
2. 加 time.Sleep 或 ticker。
3. 使用条件变量或其他同步机制。
4. 重新设计生产消费模型。

---

# 二十三、Go 并发一速记版

## 【文本块 60】goroutine 速记

goroutine 是 Go runtime 管理的轻量级并发执行单元。
goroutine 不是 OS thread，但最终运行在 OS thread 上。
goroutine 初始栈小，可动态增长。
goroutine 创建和切换成本比线程低。
main 函数退出，整个程序退出，不会自动等其他 goroutine。
goroutine 不是免费资源，过多会增加内存、调度和 GC 压力。
启动 goroutine 必须考虑生命周期、退出条件、错误处理和 panic recover。
goroutine 泄漏常见原因是 channel 阻塞、context 未取消、range channel 未关闭、网络调用无超时。

一句话：

goroutine 让并发表达变得很便宜，但工程上必须控制生命周期和并发度，不能无脑创建。

---

## 【文本块 61】GMP 速记

G 是 goroutine，表示任务。
M 是 machine，表示 OS thread。
P 是 processor，表示调度上下文和本地运行队列。
M 必须绑定 P 才能执行 Go 代码。
GOMAXPROCS 控制 P 的数量。
新 G 优先进入当前 P 的本地队列。
本地队列空了会查全局队列，也会从其他 P 偷任务，这叫 work stealing。
M 阻塞在系统调用时，P 会交给其他 M，这叫 handoff。
网络 IO 阻塞时，G 挂到 netpoller，M 不会长期阻塞。
现代 Go 支持异步抢占，避免长时间运行的 goroutine 霸占 P。

一句话：

GMP 的本质是用 runtime 调度器把大量 goroutine 高效复用到有限数量的 OS thread 上执行。

---

## 【文本块 62】channel 速记

channel 是 goroutine 之间通信和同步的机制。
无缓冲 channel 没有容量，发送和接收必须配对，强调同步交接。
有缓冲 channel 有队列，容量范围内可以解耦生产者和消费者。
channel 底层是 hchan，有环形缓冲区、sendq、recvq 和锁。
关闭 channel 表示不会再发送新数据。
接收已关闭 channel 会读到剩余数据；读完后返回零值和 ok=false。
向已关闭 channel 发送会 panic。
重复 close 会 panic。
nil channel 读写永久阻塞，close nil channel 会 panic。
for range channel 会一直读到 channel 关闭。
select 可以等待多个 channel，多个 case 就绪时伪随机选择。
default 可以做非阻塞操作，但小心忙等。
channel 是并发安全的，但不要把它当成万能锁。

一句话：

channel 适合表达数据流动、任务交接、事件通知和同步关系；mutex 更适合保护共享状态。

---

# 二十四、本章最终面试回答模板

## 【文本块 63】综合回答模板

如果面试官让我讲 Go 并发模型，我会先从 goroutine、GMP、channel 三层回答。

goroutine 是 Go runtime 管理的轻量级并发执行单元。它不是操作系统线程，但最终会被调度到 OS thread 上运行。相比直接创建线程，goroutine 初始栈更小，可以动态增长，创建和切换成本更低，所以 Go 可以用大量 goroutine 表达并发任务。但 goroutine 不是免费资源，过多会带来内存、调度和 GC 压力，也可能因为 channel 阻塞、context 未取消、网络调用无超时等原因造成泄漏。

Go runtime 底层使用 GMP 调度模型。G 表示 goroutine，M 表示 OS thread，P 表示 processor，也就是调度上下文和本地运行队列。M 必须绑定 P 才能执行 Go 代码。新创建的 goroutine 优先进入当前 P 的本地队列，本地队列空了会从全局队列拿，或者从其他 P 偷任务，这就是 work stealing。如果 M 阻塞在系统调用里，runtime 会把 P 移交给其他 M，避免调度资源被浪费。网络 IO 则通过 netpoller 管理，goroutine 等待网络事件时会被挂起，不会长期占用 OS thread。

channel 是 Go 提供的 goroutine 通信机制。无缓冲 channel 强调发送和接收同步交接，有缓冲 channel 则通过固定容量队列解耦生产者和消费者。channel 底层是 hchan，内部有环形缓冲区、发送等待队列、接收等待队列和互斥锁。close channel 表示不再发送新数据，接收方可以通过 comma ok 判断 channel 是否关闭。向已关闭 channel 发送和重复 close 都会 panic。nil channel 的读写会永久阻塞。select 可以同时监听多个 channel，常用于超时、取消、非阻塞收发和多路复用。

工程上，我不会把 goroutine 和 channel 当成万能工具。如果核心是任务交接、数据流动、pipeline、生产者消费者，我会优先使用 channel。如果核心是保护共享 map、缓存、计数器等状态，我会使用 mutex 或 RWMutex。并发代码最重要的是控制生命周期、并发度、错误传播和取消机制，而不是无限制地启动 goroutine。

一句话总结：Go 并发的优势在于用 goroutine 表达并发，用 GMP 调度器隐藏线程管理复杂度，用 channel 表达通信和同步。但真正写好 Go 并发，关键是避免 goroutine 泄漏、控制并发度，并根据场景正确选择 channel、锁和 context。
